In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 5000

# Seasonal weighting
dates = pd.date_range("2023-01-01", "2023-12-31")
weights = np.where(dates.month.isin([1,2,12]), 1.4, 1)
weights = weights / weights.sum()
admission_dates = np.random.choice(dates, size=n, p=weights)

# Introduce a few future bad dates
future_indices = np.random.choice(range(n), size=20, replace=False)
admission_dates[future_indices] = pd.Timestamp("2025-01-01")

# Admission type messy formatting
admission_type = np.random.choice(
    ["Emergency", "Elective", "emergency", "ELECTIVE"],
    size=n,
    p=[0.5, 0.2, 0.18, 0.12]
)

# Age distribution with some unrealistic values
ages = np.random.normal(loc=67, scale=20, size=n).astype(int)
ages[np.random.choice(range(n), 15)] = 150
ages[np.random.choice(range(n), 15)] = -5

# More wards + messy spacing
wards = np.random.choice(
    ["Medical", "Surgical ", "Orthopaedics", "Cardiology",
     "Elderly Care", "Respiratory", "Stroke Unit", None],
    size=n
)

# Base LOS logic (not saved)
base_los = np.random.gamma(shape=2, scale=2.5, size=n)
age_factor = (ages > 75) * 2
emergency_factor = np.array(
    [1 if str(x).lower() == "emergency" else 0 for x in admission_type]
) * 2

los = (base_los + age_factor + emergency_factor).astype(int) + 1
# Create discharge dates
discharge_dates = (
    pd.to_datetime(admission_dates, errors='coerce') +
    pd.to_timedelta(los, unit="D")
)

# Convert to numpy array so we can edit
discharge_dates = discharge_dates.to_numpy()

# Introduce missing discharge dates
missing_indices = np.random.choice(range(n), size=100, replace=False)
discharge_dates[missing_indices] = np.datetime64("NaT")

# Introduce some incorrect discharge before admission
error_indices = np.random.choice(range(n), size=20, replace=False)
discharge_dates[error_indices] = (
    pd.to_datetime(admission_dates[error_indices]) -
    pd.to_timedelta(3, unit="D")
)

# Readmission flag
readmission_prob = np.where(los > 7, 0.15, 0.08)
readmission_30d = np.random.binomial(1, readmission_prob)

df = pd.DataFrame({
    "patient_id": np.random.choice(range(1, 4800), size=n),  # duplicates
    "age": ages,
    "admission_date": pd.to_datetime(admission_dates),
    "discharge_date": discharge_dates,
    "admission_type": admission_type,
    "ward": wards,
    "readmission_30d": readmission_30d
})

df.to_csv("hospital_admissions_messy.csv", index=False)

print("Messy dataset created.")

Messy dataset created.
